# Set up

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import re, os, sqlite3
from datetime import datetime

# Function to extract score from options based on values in the response
def val_score_mapping(s1,s2):
    
    split_options = s2.strip().split("),")
    split_response = s1.strip().split(": ")[1].split(',')
    scores = {} 

    for i in split_options:
        if len(i) > 0:
            val_num = i.split("(score")[0].split(': ')[1].strip()
            score_num = i.split("(score")[1].split(': ')[1].strip()
            scores[val_num] = score_num

    response_score_mapping = {split_response[i].strip(): scores[split_response[i].strip()] for i in range(len(split_response))}
    list_response_score_mapping = list(response_score_mapping.values())
    str_response_score_mapping = ', '.join(str(value) for value in list_response_score_mapping)
    return str_response_score_mapping.replace(')', '')

# Function to cleanup and split time range in the response
def clean_time_range(df, column_name):
    cleaned = [] 
    for i in range(len(df[column_name])):
        if pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith('time_range'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            # tpos = datetime.strptime(str(ttemp), '%H:%M')
            # thm = tpos.strftime('%H:%M')
            thm = ttemp      
        elif pd.notna(df[column_name][i]) and str(df[column_name][i]).startswith(' to'):
            t = re.sub(r'[a-zA-Z\s+(\)_:]', '', df[column_name][i])
            t = t.replace(',', ':')
            if re.search(r'^[0-9]:', t): #9,30/12,30
                ttemp = '0' + t
            elif re.search(r':[0-9]$', t): #12,5/12,30
                ttemp = t.replace(':', ':0')
            else:
                ttemp = t
            thm = ttemp      
        else:
            ttemp = df[column_name][i] 
            thm = ttemp
        cleaned.append(thm)
    return cleaned

#### Read in data

In [2]:
# Set up path variable and get list of responses.csv files

input_path = os.path.expanduser('~/NIMH EMA Data/Input Files/') #home directory > replace with your EMA folder > new folder called Input Files
all_files = os.listdir(os.path.join(input_path, 'EMA_applet_data'))
files = [file for file in all_files if file.startswith('responses')]
input_files = os.listdir(input_path)
output_path = os.path.expanduser('~/NIMH EMA Data/Output Files/')

# Read all the responses.csv files
responses_all = []
for i in range(len(files)):
    temp_df = pd.read_csv(os.path.join(input_path, 'EMA_applet_data', files[i]), encoding='ISO-8859-1')
    responses_all.append(temp_df)

# Concat responses.csv to one file
dat_full = pd.concat(responses_all, ignore_index=True)

dat_full = dat_full.map(str)

# Write out the concatenated responses.csv file
dat_full.to_csv(os.path.join(output_path,'responses_all.csv'),index=False)

# Data Cleaning

In [3]:
# Rename target_id column as it contains special characters
dat_full.rename(columns={'ï»¿target_id': 'target_id'}, inplace=True) 

# Add utc_timezone_offset
dat_full['offset'] = np.where(dat_full['utc_timezone_offset']=='nan', 0, 1)

dat_full['activity_start_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_start_time'], errors='coerce')+ (pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_start_time'], errors='coerce'))
dat_full['activity_end_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_end_time'], errors='coerce')+ (pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_end_time'], errors='coerce'))
dat_full['activity_schedule_start_time_offsetADD'] = np.where(dat_full['utc_timezone_offset'] != 'nan', pd.to_numeric(dat_full['activity_schedule_start_time'], errors='coerce')+(pd.to_numeric(dat_full['utc_timezone_offset'], errors='coerce')* 60 * 1000) , pd.to_numeric(dat_full['activity_schedule_start_time'], errors='coerce'))

dat_full['activity_start_time_offsetADD'] = pd.to_numeric(dat_full['activity_start_time_offsetADD'], downcast='integer')
dat_full['activity_end_time_offsetADD'] = pd.to_numeric(dat_full['activity_end_time_offsetADD'], downcast='integer')
dat_full['activity_schedule_start_time_offsetADD'] = pd.to_numeric(dat_full['activity_schedule_start_time_offsetADD'], downcast='integer')

dat_full['start_Time'] = dat_full['activity_start_time_offsetADD']
dat_full['end_Time'] = dat_full['activity_end_time_offsetADD']
dat_full['schedule_Time'] = dat_full['activity_schedule_start_time_offsetADD']

In [4]:
# Separate activity (activity watch, light sensor) and flow (EMA) data to process individually, then re-merge

dat_act = dat_full[(dat_full['activity_name'] == 'Light Device') | (dat_full['activity_name'] == 'Activity Watch')] # activity (device) data
dat_rest = dat_full[(dat_full['activity_name'] != 'Light Device') & (dat_full['activity_name'] != 'Activity Watch')] # flow (EMA) data

# For EMA data: adjusting start and end times 
dat_rest = dat_rest.groupby(['secret_user_id', 'activity_flow_id', 'activity_schedule_start_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)

# For device data: make schedule_Time = start_Time (since they do not have scheduled times)
for index, row in dat_act.iterrows():
    dat_act.at[index, 'schedule_Time'] = row['start_Time']

# Re-merge activity and flow data
dat_processed = pd.merge(dat_act, dat_rest, on=['target_id', 'target_secret_id', 'target_nickname', 'target_tag', 'source_id', 'source_secret_id', 'source_nickname', 'source_tag', 'source_relation', 'input_id', 'input_secret_id', 'input_nickname', 'userId', 'secret_user_id', 'legacy_user_id', 'applet_version', 'activity_flow_id', 'activity_flow_name', 'activity_flow_submission_id', 'activity_id', 'activity_name', 'activity_submission_id', 'activity_start_time', 'activity_end_time', 'activity_schedule_id', 'activity_schedule_start_time', 'utc_timezone_offset', 'activity_submission_review_id', 'item_id', 'item_name', 'item_prompt', 'item_response_options', 'item_response', 'item_response_status', 'rawScore', 'offset', 'activity_start_time_offsetADD', 'activity_end_time_offsetADD', 'activity_schedule_start_time_offsetADD', 'start_Time', 'end_Time', 'schedule_Time'], how='outer')

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/2876284611.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dat_rest = dat_rest.groupby(['secret_user_id', 'activity_flow_id', 'activity_schedule_start_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)


In [5]:
# Employ cleaning code provided by Curious team

dat_subset = dat_processed[['activity_submission_id', 'activity_schedule_start_time', 'secret_user_id', 'userId',
       'activity_id', 'activity_name', 'activity_flow_id', 'activity_flow_name', 'item_name', 'item_response', 'item_response_options',
       'activity_schedule_id', 'start_Time', 'end_Time', 'schedule_Time', 'applet_version', 'activity_start_time', 'offset']]

# Create additional column to add scores in the next steps
dat_subset = dat_subset.copy()
dat_subset['item_response_scores'] = None

# Cleanup
for i in range(len(dat_subset['item_response'])):
    if re.search(r'score: ', dat_subset['item_response_options'][i]):
        s = val_score_mapping(dat_subset['item_response'][i], dat_subset['item_response_options'][i])
        dat_subset.loc[i, 'item_response_scores'] = s  
    if re.search(r'value', dat_subset['item_response'][i]):
        r = dat_subset['item_response'][i].replace("value: ", "")
        dat_subset.loc[i, 'item_response'] = r
    elif re.search(r'time:', dat_subset['item_response'][i]):
        if re.search(r'hr [0-9],', dat_subset['item_response'][i]): 
            egapp = dat_subset['item_response'][i]. replace('time: hr ', '0')
            if re.search(r', min [0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':0')
            elif re.search(r', min [0-9][0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':')
            egpos = datetime.strptime(egtemp, '%H:%M')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'item_response'] = egpos2
        elif re.search(r'hr [0-9][0-9],', dat_subset['item_response'][i]):
            egapp = dat_subset['item_response'][i].replace('time: hr ', '')
            if re.search(r', min [0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':0')
            elif re.search(r', min [0-9][0-9]$', egapp): 
                egtemp = egapp.replace(', min ', ':')
            egpos = datetime.strptime(egtemp, '%H:%M')
            egpos2 = egpos.strftime('%H:%M')
            dat_subset.loc[i, 'item_response'] = egpos2
    elif re.search(r'geo:', dat_subset['item_response'][i]):
        g = dat_subset['item_response'][i].replace('geo: ', '')
        dat_subset.loc[i, 'item_response'] = g

# Combine scores and other formats of responses into one column
dat_subset['item_response2'] = np.where(dat_subset['item_response_scores'].isna(), dat_subset['item_response'], dat_subset['item_response_scores'])

# Sort and Select required columns
dat_subset = dat_subset.sort_values(by=['secret_user_id', 'activity_flow_id', 'activity_id', 'activity_name', 'schedule_Time', 'activity_start_time'])

### Additional Cleaning

In [6]:
# Create a new binary to indicate whether responses were from activity or flow
dat_subset['is_activity'] = np.where(dat_subset['activity_flow_id'] == 'nan', 'Y', 'N')

# Create a new column such that:
#   activity_flow_id if item is from EMA
#   activity_id if item is from activity (device) log
dat_subset['activity_flow'] = np.where(dat_subset['activity_flow_id'] == 'nan', (dat_subset['activity_id'] + '|' + dat_subset['activity_name']), dat_subset['activity_flow_id'])
dat_subset = dat_subset[['userId', 'secret_user_id', 'activity_flow_id', 'activity_id', 'activity_flow', 'activity_flow_name', 'is_activity', 'offset', 'item_name', 'item_response', 
                         'item_response_scores', 'item_response2','item_response_options', 'start_Time',
                         'end_Time', 'schedule_Time', 'activity_schedule_start_time', 'applet_version', 'activity_submission_id', 'activity_schedule_id']]
 
# Make sure there are no NAs 
dat_subset['schedule_Time'] = np.where(dat_subset['schedule_Time'].isna(), 'NO SCHEDULE', dat_subset['schedule_Time'])

### Widening Data

In [7]:
# Add answer_ids to help with debugging in the final output
answers = dat_subset.groupby(['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'])['activity_submission_id'].apply(lambda x: '|'.join(x.astype(str))).reset_index()

# Widen data
dat_wide = pd.pivot_table(dat_subset, index=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], columns='item_name', values='item_response2', aggfunc='last').reset_index()

# Join Wide format table with answers to include concatenated answer_ids
dat_wide = pd.merge(dat_wide, answers, on=['userId', 'secret_user_id', 'activity_flow', 'activity_flow_name', 'activity_schedule_id', 'is_activity', 'start_Time', 'end_Time', 'schedule_Time', 'offset', 'applet_version'], how='outer')

dat_wide.head()

,userId,secret_user_id,activity_flow,activity_flow_name,activity_schedule_id,is_activity,start_Time,end_Time,schedule_Time,offset,...,since_substances_other_tobacco_time,since_times_eat,since_vigorous_activity,since_where,socialmedia_activity,socialmedia_duration,strangers_communication_method,substances_tobacco,videogame_duration,activity_submission_id
0,042cdf4a-3d66-4523-84ee-8a61fff2fd3d,nw01,38d4c92e-d1fc-4e55-bf07-421d4104b4f5|Activity ...,nan,13edde07-7fb4-4139-a009-dae7d486fa7b,Y,1756310695345,1756310700595,1756310695345.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bab54d22-d81c-481f-ac57-3a821e333978
1,042cdf4a-3d66-4523-84ee-8a61fff2fd3d,nw01,503cf745-49af-40bb-add9-4936124db2ea,Afternoon Assessment,b86587e9-cdaa-4471-8d92-75ccc72d6a29,N,1756310617499,1756313999000,1756310400000.0,1,...,NaN,NaN,NaN,"2, 1",NaN,NaN,NaN,NaN,NaN,be3c7729-c759-468e-a007-fefa07dff6c5|be3c7729-...
2,2cfcd773-16d8-45bf-9d43-0eb06fb46090,1000-111,3402e0b4-f306-400b-ac81-a314ee72efaf,Evening Assessment,f9e8d1e8-3f1d-4e9b-8de8-966bc6584112,N,1774121450718,1774121751026,1774121400000.0,1,...,NaN,0,0,1,NaN,NaN,NaN,NaN,NaN,7c6a2260-e35e-442f-8782-cd1746a0445d|7c6a2260-...
3,2cfcd773-16d8-45bf-9d43-0eb06fb46090,1000-111,3402e0b4-f306-400b-ac81-a314ee72efaf,Evening Assessment,f9e8d1e8-3f1d-4e9b-8de8-966bc6584112,N,1774208340428,1774208474021,1774207800000.0,1,...,NaN,0,0,1,NaN,NaN,NaN,NaN,NaN,7bfd563a-449d-4b22-b40f-1e98de434b2c|7bfd563a-...
4,2cfcd773-16d8-45bf-9d43-0eb06fb46090,1000-111,3402e0b4-f306-400b-ac81-a314ee72efaf,Evening Assessment,f9e8d1e8-3f1d-4e9b-8de8-966bc6584112,N,1774296308241,1774296457280,1774294200000.0,1,...,NaN,0,0,1,NaN,NaN,NaN,NaN,NaN,ae82ea7f-5837-4fb2-ab87-82be661e6802|ae82ea7f-...


### Specific item cleaning

In [8]:
# Use the function from the Set Up section to split headache_time_range column into start and end times
headache_times = ['headache_time_start', 'headache_time_end']

for i in headache_times: 
    if i in dat_wide.columns:
        dat_wide[i] = clean_time_range(dat_wide, i)
        dat_wide[i] = clean_time_range(dat_wide, i)

# Clean up gps response and split to 2 columns for latitude and longitude
if 'now_gps' in dat_wide.columns:
    dat_wide['now_gps'] = dat_wide['now_gps'].replace(r'[a-zA-Z\s+(\)]', '', regex = True)
    dat_wide[['now_gps_lat', 'now_gps_long']] = dat_wide['now_gps'].str.split("/", expand=True)

# Use the function from the Set Up section to split since_activity_monitor_time and since_light_device_time columns into start and end times
devices_time_range = ['since_activity_monitor_time', 'since_light_device_time']

for i in devices_time_range: 
    if i in dat_wide.columns:
        start = i + '_start'
        end = i + '_end'
        dat_wide[[start, end]] = dat_wide[i].str.split("/", expand=True)
        dat_wide[start] = clean_time_range(dat_wide, start)
        dat_wide[end] = clean_time_range(dat_wide, end) 

### Timestamp cleaning

In [9]:
dat_wide_full = dat_wide.map(str)

# Epoch to Timestamp
dat_wide_full['start_Time'] = pd.to_numeric(dat_wide_full['start_Time'], errors='coerce')
dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'] / 1000, unit='s')

dat_wide_full['end_Time'] = pd.to_numeric(dat_wide_full['end_Time'], errors='coerce')
dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'] / 1000, unit='s')

dat_wide_full['schedule_Time'] = pd.to_numeric(dat_wide_full['schedule_Time'], errors='coerce')
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'] / 1000, unit='s')

# Timestamp cleanup 
dat_wide_full['schedule_Time'] = pd.to_datetime(dat_wide_full['schedule_Time'])
dat_wide_full['schedule_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['schedule_Time'],  dat_wide_full['schedule_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None)) 

dat_wide_full['start_Time'] = pd.to_datetime(dat_wide_full['start_Time'])
dat_wide_full['start_Time'] = dat_wide_full['start_Time'].dt.floor('1s')
dat_wide_full['start_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['start_Time'], dat_wide_full['start_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None))

dat_wide_full['end_Time'] = pd.to_datetime(dat_wide_full['end_Time'])
dat_wide_full['end_Time'] = dat_wide_full['end_Time'].dt.floor('1s')
dat_wide_full['end_Time'] = np.where(dat_wide_full['offset']== '1', dat_wide_full['end_Time'], dat_wide_full['end_Time'].dt.tz_localize('UTC').dt.tz_convert('America/New_York').dt.tz_localize(None))

# Creating Dates
dat_wide_full['scheduled_Date'] = dat_wide_full['schedule_Time'].dt.date
dat_wide_full['start_Date'] = dat_wide_full['start_Time'].dt.date

### Split data into flows and activities

In [10]:
act_dat_wide = dat_wide_full[dat_wide_full['is_activity']=='Y'] # activity (device) data
flow_dat_wide = dat_wide_full[dat_wide_full['is_activity']=='N'] # flow (EMA) data

# Flow Submissions (EMA only)

In [11]:
# Timestamp formatting - TO MAKE SURE!
flow_dat_wide['schedule_Time'] = pd.to_datetime(flow_dat_wide['schedule_Time'], format='mixed')
flow_dat_wide['start_Time'] = pd.to_datetime(flow_dat_wide['start_Time'], format="mixed")
flow_dat_wide['end_Time'] = pd.to_datetime(flow_dat_wide['end_Time'])

final = flow_dat_wide

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/1239275528.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flow_dat_wide['schedule_Time'] = pd.to_datetime(flow_dat_wide['schedule_Time'], format='mixed')
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/1239275528.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  flow_dat_wide['start_Time'] = pd.to_datetime(flow_dat_wide['start_Time'], format="mixed")
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/12392

### Flows Final

In [ ]:
#################################################################### PLEASE SELECT APPROPRIATE ITEMS & ORDER ###############################################################

# Specify the required columns, in desired order, for the final output

cols = [
    'userId',
    'activity_schedule_id',
    'activity_submission_id',
    'secret_user_id',
    'activity_flow_name',
    'schedule_Time',
    'start_Time',
    'end_Time',
    'applet_version',
    'saliva_instructions',
    'since_activity_monitor',
    'since_activity_monitor_time_start', 
    'since_activity_monitor_time_end', 
    'since_light_device',
    'since_light_device_time_start', 
    'since_light_device_time_end', 
    'morning_sleep_quantity',
    'morning_bedtime',
    'morning_lights_off',
    'morning_fall_asleep',
    'morning_wake_number',
    'morning_awakenings_last',
    'morning_waketime',
    'morning_outbed',
    'morning_sleep_quality',
    'morning_sleep_refreshed',
    'morning_sleep_problems',
    'morning_sleeping_pills',
    'morning_sleeping_pills_type',
    'now_gps_lat', 
    'now_gps_long', 
    'now_where',
    'now_inside',
    'now_outside',
    'since_where',
    'since_inside',
    'since_outside',
    'now_company',
    'since_company',
    'now_activity',
    'since_activity',
    'since_internet',
    'internet_use_duration',
    'internet_use_category',
    'ai_chatbot_duration',
    'ai_chatbot_category',
    'ai_chatbot_category_other',
    'videogame_duration',
    'socialmedia_duration',
    'socialmedia_activity',
    'friends_communication_method',
    'strangers_communication_method',
    'comments_byothers',
    'comments_byself',
    'now_sadness',
    'now_anxiousness',
    'now_active',
    'now_tired',
    'now_distracted',
    'now_irritable',
    'now_quick_thinking',
    'now_enjoyment',
    'now_fidgety',
    'now_thoughts_positive',
    'now_thoughts_negative',
    'now_thoughts_negative_about',
    'now_thoughts_negative_severity',
    'since_thoughts_negative',
    'since_thoughts_negative_suicide',
    'since_thoughts_negative_suicide_message',
    'instructions', #event_instructions
    'event_emotion',
    'event_category',
    'event_people',
    'event_where',
    'event_health',
    'event_content',
    'event_issue',
    'event_stress',
    'since_physical_activity',
    'since_sedentary',
    'since_nap_rest_time',
    'since_rest_duration',
    'since_rest_fell_asleep',
    'since_sleep_duration',
    'since_vigorous_activity',
    'since_moderate_activity',
    'since_light_activity',
    'now_thirsty',
    'since_had_drink',
    'not_alcohol_amount',
    'since_had_drink_alcohol_type',
    'since_had_drink_alcohol_quantity',
    'alcohol_time',
    'since_feel_drink',
    'since_had_drink_caffeinated_type',
    'now_hungry',
    'since_times_eat',
    'since_eaten_amount',
    'since_eaten_when',
    'since_eaten_type',
    'since_food1',
    'since_food2',
    'since_food3',
    'since_food4',
    'since_food5',
    'since_crave',
    'craving_strong_tobacco',
    'craving_strong_cannabis',
    'craving_strong_otherdrug',
    'craving_strong_alcohol',
    'since_substances',
    'substances_tobacco',
    'since_substances_cigarettes',
    'since_substances_cigarettes_time',
    'since_substances_enicotine',
    'since_substances_enicotine_time',
    'since_substances_other_tobacco',
    'since_substances_other_tobacco_time',
    'since_cannabis_type',
    'since_cannabis_time',
    'since_substances_cannabis',
    'since_cannabis_high',
    'since_substances_other',
    'since_substances_other_specify',
    'since_substances_other_time',
    'since_substances_other_high',
    'since_substances_company',
    'now_pain',
    'now_pain_where',
    'since_pain',
    'since_pain_where',
    'pain_intensity',
    'headache',
    'headache_prevent', 
    'headache_same',
    'headache_time_start', 
    'headache_current', 
    'headache_time_end', 
    'headache_intensity',
    'headache_location',
    'headache_pulsating',
    'headache_effort',
    'headache_nausea',
    'headache_light',
    'headache_heat',
    'headache_noise',
    'headache_smell',
    'headache_trigger',
    'headache_vision_changes',
    'headache_vision_change_time',
    'headache_numbing',
    'headache_numbing_time',
    'headache_confusing',
    'headache_confusing_time',
    'headache_sudden',
    'headache_medication',
    'headache_interference',
    'day_physical_health',
    'day_problem_categories',
    'day_bother',
    'day_over_medication',
    'day_over_medication_why',
    'day_prescribed_medication',
    'day_prescribed_medication_conditions',
    'day_problems_allergies',
    'day_problems_breath',
    'day_problems_belly_symptoms',
    'day_problems_belly',
    'day_problems_muscle',
    'day_problems_heart',
    'dizziness_situation',
    'dizziness_faint',
    'day_lethargic',
    'activity_planned',
    'day_period'
    ]

# Select and arrange columns, adding manual cols as NA if needed (to ensure all variables are displayed)
final = final.reindex(columns=cols)

# Sort by user ID and time of assessment
final = final.sort_values(by=['secret_user_id', 'schedule_Time', 'start_Time'])

In [13]:
# Ensure consistent NA wording across all columns
final_out = final.copy()
final_out.replace('nan', 'NA', inplace=True)
final_out.fillna('NA', inplace=True)
final_out.replace('NA NA', 'NA', inplace=True)
final_out.rename(columns = {'userId_x':'user_id'},inplace=True)

# Exclude test flows
final_out = final_out[final_out['activity_flow_name'] != 'Test Flow (All Activities)']

# Check final flow data
final_out.head()

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/729725924.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_out.fillna('NA', inplace=True)


,userId,activity_schedule_id,activity_submission_id,secret_user_id,activity_flow_name,schedule_Time,start_Time,end_Time,applet_version,saliva_instructions,...,day_problems_breath,day_problems_belly_symptoms,day_problems_belly,day_problems_muscle,day_problems_heart,dizziness_situation,dizziness_faint,day_lethargic,activity_planned,day_period
10,2cfcd773-16d8-45bf-9d43-0eb06fb46090,72035b43-8ce2-41fd-8639-f23f4796583d,3744483c-b706-45f4-9e1a-28e4ff849901|3744483c-...,1000-111,Morning Assessment,2026-03-21 06:30:00,2026-03-21 06:30:19,2026-03-21 06:33:26,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
13,2cfcd773-16d8-45bf-9d43-0eb06fb46090,b7ba459d-a7b4-4b3f-bc66-b5c284156279,1d209f3c-0baf-42f0-a753-8abf46ed38a0|1d209f3c-...,1000-111,Mid-day Assessment,2026-03-21 13:30:00,2026-03-21 13:32:45,2026-03-21 13:34:48,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
6,2cfcd773-16d8-45bf-9d43-0eb06fb46090,6fa92c24-e4ff-441f-a133-45844312ea93,fcd2b48e-623a-44de-8963-0bee4ff5b0da|fcd2b48e-...,1000-111,Afternoon Assessment,2026-03-21 16:30:00,2026-03-21 16:34:23,2026-03-21 16:41:12,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,2cfcd773-16d8-45bf-9d43-0eb06fb46090,f9e8d1e8-3f1d-4e9b-8de8-966bc6584112,7c6a2260-e35e-442f-8782-cd1746a0445d|7c6a2260-...,1000-111,Evening Assessment,2026-03-21 19:30:00,2026-03-21 19:30:50,2026-03-21 19:35:51,1.1.0,NA,...,4,"1, 3, 2, 4",4,4,4,1,1,1,1,NA
7,2cfcd773-16d8-45bf-9d43-0eb06fb46090,6fa92c24-e4ff-441f-a133-45844312ea93,13c112fc-639f-4508-af58-cde5d1d9c73a|13c112fc-...,1000-111,Afternoon Assessment,2026-03-22 16:30:00,2026-03-22 17:00:42,2026-03-22 17:04:23,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [14]:
# Write out final flow dataframe: necessary for secondary QC script.

final_out.to_csv(os.path.join(output_path, 'flow_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

# Activity Submissions (devices only)

In [15]:
# Check your ACTIVITIES Submissions data:
# Will only have data from activities like activity watch or light sensors logs, otherwise dataset below will be empty. 
# If empty, you can skip the rest of the script.

act_dat_wide.head()

,userId,secret_user_id,activity_flow,activity_flow_name,activity_schedule_id,is_activity,start_Time,end_Time,schedule_Time,offset,...,socialmedia_activity,socialmedia_duration,strangers_communication_method,substances_tobacco,videogame_duration,activity_submission_id,now_gps_lat,now_gps_long,scheduled_Date,start_Date
0,042cdf4a-3d66-4523-84ee-8a61fff2fd3d,nw01,38d4c92e-d1fc-4e55-bf07-421d4104b4f5|Activity ...,nan,13edde07-7fb4-4139-a009-dae7d486fa7b,Y,2025-08-27 16:04:55,2025-08-27 16:05:00,2025-08-27 16:04:55.345000029,1,...,nan,nan,nan,nan,nan,bab54d22-d81c-481f-ac57-3a821e333978,nan,nan,2025-08-27,2025-08-27


In [16]:
# Timestamp formatting - TO MAKE SURE!
act_dat_wide['schedule_Time'] = pd.to_datetime(act_dat_wide['schedule_Time'], format="mixed")
act_dat_wide['start_Time'] = pd.to_datetime(act_dat_wide['start_Time'], format = "mixed")
act_dat_wide['end_Time'] = pd.to_datetime(act_dat_wide['end_Time'], format = "mixed")

act_final = act_dat_wide

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/3808806946.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  act_dat_wide['schedule_Time'] = pd.to_datetime(act_dat_wide['schedule_Time'], format="mixed")
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/3808806946.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  act_dat_wide['start_Time'] = pd.to_datetime(act_dat_wide['start_Time'], format = "mixed")
/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/3808806

### Activities Final

In [17]:
################################################################################### PLEASE SELECT APPROPRIATE ITEMS & ORDER ############################################################################

# Uses the same "cols" list provided above to specify the required columns, in desired order, for the final output

# Select and arrange columns, adding manual cols as NA if needed (to ensure all variables are displayed)
act_final = act_final.reindex(columns=cols)

# Sort by user ID and time of assessment
act_final = act_final.sort_values(by=['secret_user_id', 'start_Time'])

In [18]:
# Ensure consistent NA wording across all columns
act_final_out = act_final.copy()
act_final_out.replace('nan', 'NA', inplace=True)
act_final_out.fillna('NA', inplace=True)
act_final_out.replace('NA NA', 'NA', inplace=True)

# Check final activity data
act_final_out.head()

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/3480809165.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  act_final_out.fillna('NA', inplace=True)


,userId,activity_schedule_id,activity_submission_id,secret_user_id,activity_flow_name,schedule_Time,start_Time,end_Time,applet_version,saliva_instructions,...,day_problems_breath,day_problems_belly_symptoms,day_problems_belly,day_problems_muscle,day_problems_heart,dizziness_situation,dizziness_faint,day_lethargic,activity_planned,day_period
0,042cdf4a-3d66-4523-84ee-8a61fff2fd3d,13edde07-7fb4-4139-a009-dae7d486fa7b,bab54d22-d81c-481f-ac57-3a821e333978,nw01,NA,2025-08-27 16:04:55.345000029,2025-08-27 16:04:55,2025-08-27 16:05:00,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [19]:
# Write out final activity dataframe. Again, will only have data from activities like watch/sensor logs
act_final_out.to_csv(os.path.join(output_path, 'activity_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

# Final Output (flow and activity together)

In [20]:
# Join flow final and activity final togeter 
# If you don't have any activity data, this will be identical to flow final and you can skip the rest of the script.

all_final = [final_out, act_final_out]
ema_out = pd.concat(all_final, ignore_index=True)

# Ensure alignment with formatting
ema_out = ema_out.applymap(str)

# Final data output sorting
ema_out = ema_out.sort_values(by=['secret_user_id', 'schedule_Time'])

/var/folders/gr/8ff1xlg55jgfk7328g8nqgyh0000gv/T/ipykernel_46161/1900246699.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  ema_out = ema_out.applymap(str)


Check final output

In [21]:
# Check for duplicates in final file. Ensure this always returns "False"
print(ema_out.duplicated().any())

False


In [22]:
# Preview final output
ema_out.head()

,userId,activity_schedule_id,activity_submission_id,secret_user_id,activity_flow_name,schedule_Time,start_Time,end_Time,applet_version,saliva_instructions,...,day_problems_breath,day_problems_belly_symptoms,day_problems_belly,day_problems_muscle,day_problems_heart,dizziness_situation,dizziness_faint,day_lethargic,activity_planned,day_period
0,2cfcd773-16d8-45bf-9d43-0eb06fb46090,72035b43-8ce2-41fd-8639-f23f4796583d,3744483c-b706-45f4-9e1a-28e4ff849901|3744483c-...,1000-111,Morning Assessment,2026-03-21 06:30:00,2026-03-21 06:30:19,2026-03-21 06:33:26,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
1,2cfcd773-16d8-45bf-9d43-0eb06fb46090,b7ba459d-a7b4-4b3f-bc66-b5c284156279,1d209f3c-0baf-42f0-a753-8abf46ed38a0|1d209f3c-...,1000-111,Mid-day Assessment,2026-03-21 13:30:00,2026-03-21 13:32:45,2026-03-21 13:34:48,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,2cfcd773-16d8-45bf-9d43-0eb06fb46090,6fa92c24-e4ff-441f-a133-45844312ea93,fcd2b48e-623a-44de-8963-0bee4ff5b0da|fcd2b48e-...,1000-111,Afternoon Assessment,2026-03-21 16:30:00,2026-03-21 16:34:23,2026-03-21 16:41:12,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,2cfcd773-16d8-45bf-9d43-0eb06fb46090,f9e8d1e8-3f1d-4e9b-8de8-966bc6584112,7c6a2260-e35e-442f-8782-cd1746a0445d|7c6a2260-...,1000-111,Evening Assessment,2026-03-21 19:30:00,2026-03-21 19:30:50,2026-03-21 19:35:51,1.1.0,NA,...,4,"1, 3, 2, 4",4,4,4,1,1,1,1,NA
4,2cfcd773-16d8-45bf-9d43-0eb06fb46090,6fa92c24-e4ff-441f-a133-45844312ea93,13c112fc-639f-4508-af58-cde5d1d9c73a|13c112fc-...,1000-111,Afternoon Assessment,2026-03-22 16:30:00,2026-03-22 17:00:42,2026-03-22 17:04:23,1.1.0,NA,...,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [23]:
# Write the final output file to csv
ema_out.to_csv(os.path.join(output_path, 'ema_output_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')

In [ ]:
# For a specific participant:
# xxx_ema = ema_out[ema_out['secret_user_id'] == '9999-999']
# xxx_ema.to_csv(os.path.join(output_path, '9999-999_ema_output_final.csv'), index=False, date_format='%Y-%m-%d %H:%M:%S', na_rep='NA')